## Part 1: Practice

### 1.

- Price is string.
- There are 3 empty columns: 'coupon_code', 'damage_notes' and 'late_fee_note'.
- After removal the dataset went from 15 columns to 12 (0 empty).

In [84]:
import pandas as pd

import numpy as np

rentals_raw = pd.read_csv("data/bike_rentals.csv")

rentals = rentals_raw.copy()

print(rentals.shape)
print(rentals.info())

(403, 15)
<class 'pandas.DataFrame'>
RangeIndex: 403 entries, 0 to 402
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   rental_id      403 non-null    str    
 1   station        403 non-null    str    
 2   bike_type      403 non-null    str    
 3   helmets_text   399 non-null    str    
 4   is_member      403 non-null    str    
 5   insured        400 non-null    str    
 6   paid_online    403 non-null    str    
 7   status         403 non-null    str    
 8   coupon_code    0 non-null      float64
 9   damage_notes   0 non-null      float64
 10  late_fee_note  0 non-null      float64
 11  start_date     403 non-null    str    
 12  end_date       403 non-null    str    
 13  price          352 non-null    str    
 14  rating         288 non-null    float64
dtypes: float64(4), str(11)
memory usage: 47.4 KB
None


In [85]:
print(len(rentals.columns[rentals.isna().all()]))
print(list(rentals.columns[rentals.isna().all()]))

3
['coupon_code', 'damage_notes', 'late_fee_note']


In [86]:
rentals = rentals.dropna(axis=1, how="all")

print(rentals.shape)
print(len(rentals.columns[rentals.isna().all()]))

(403, 12)
0


### 2.

In [87]:
rentals["price"].head(10)

0      SEK 855.50
1    SEK 1,311.50
2      SEK 350.00
3      SEK 569.50
4      SEK 828.00
5      SEK 246.00
6      SEK 476.00
7    SEK 1,838.50
8    SEK 1,555.00
9      SEK 189.50
Name: price, dtype: str

In [88]:
rentals["price"] = rentals["price"].str.replace("SEK", "").str.strip()
rentals["price"].head(3)

0      855.50
1    1,311.50
2      350.00
Name: price, dtype: str

In [89]:
rentals["price"] = rentals["price"].str.replace(",", "")
rentals["price"].head(3)

0     855.50
1    1311.50
2     350.00
Name: price, dtype: str

In [90]:
rentals["price"] = rentals["price"].astype(float)

print(rentals["price"].dtype)
rentals["price"].describe().round(2)

float64


count     352.00
mean      847.35
std       633.44
min       154.50
25%       406.00
50%       668.25
75%      1194.62
max      3401.00
Name: price, dtype: float64

In [91]:
rentals.sort_values("price", ascending=False).head(3)[["rental_id", "bike_type", "status", "price"]]

,rental_id,bike_type,status,price
187,R-5282,Cargo bike,Completed,3401.0
49,R-5209,Cargo bike,Completed,2979.5
270,R-5378,E-bike,Completed,2941.0


### 3.

- 51 prices are missing. About 12.7%
- 47 of the missing prices can be explained by the rental being cancelled wheras 4 of the missing prices cannot be explained this way.
- There isn't anything else in these 4 rows that explained why the price is missing either.
- I will leave all the missing prices as is since it is of no practical use to put in a number. Also, any added number would make price statistics less accurate.

In [92]:
print(rentals["price"].isna().sum())
print((rentals["price"].isna().mean() * 100).round(2))

51
12.66


In [93]:
rentals["status"].value_counts()

status
Completed    334
Cancelled     47
No-show       22
Name: count, dtype: int64

In [94]:
no_price = rentals["price"].isna()
cancelled = rentals["status"] == "Cancelled"

pd.crosstab(no_price, cancelled)

status,False,True
price,,
False,352,0
True,4,47


In [95]:
rentals[no_price][~cancelled]

C:\Users\chris\AppData\Local\Temp\ipykernel_10304\910778322.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  rentals[no_price][~cancelled]


,rental_id,station,bike_type,helmets_text,is_member,insured,paid_online,status,start_date,end_date,price,rating
68,R-5068,Kungsholmen,City bike,1 helmet,f,f,t,Completed,2026-08-07,2026-08-13,NaN,4.0
108,R-5377,Kungsholmen,City bike,2 helmets,f,t,f,Completed,2026-08-13,2026-08-16,NaN,5.0
192,R-5375,Hammarby,E-bike,1 helmet,t,f,t,Completed,2026-03-24,2026-03-25,NaN,3.0
241,R-5231,Östermalm,City bike,2 helmets,f,NaN,t,Completed,2026-07-19,2026-07-20,NaN,NaN


### 4.

- The columns with t and f are 'is_member', 'insured' and 'paid_online'.
- 'status' is not in the list since it contains the values: Completed, Cancelled and No-show
- The only column to have missing values after changing t and f to True and False is 'insured' with 3 missing values
- The members have a lower median-price (558) than the non-members(756)

In [96]:
tf_columns = [col for col in rentals.columns if set(rentals[col].dropna().unique()) <= {"t", "f"}]

print(tf_columns)

['is_member', 'insured', 'paid_online']


In [97]:
rentals["status"].value_counts()

status
Completed    334
Cancelled     47
No-show       22
Name: count, dtype: int64

In [98]:
rentals["is_member"].value_counts()

is_member
f    288
t    115
Name: count, dtype: int64

In [99]:
rentals["insured"].value_counts()

insured
t    219
f    181
Name: count, dtype: int64

In [100]:
rentals["paid_online"].value_counts()

paid_online
t    292
f    111
Name: count, dtype: int64

In [102]:
rentals["is_member"] = rentals["is_member"].map({"t": True, "f": False})
rentals["insured"] = rentals["insured"].map({"t": True, "f": False})
rentals["paid_online"] = rentals["paid_online"].map({"t": True, "f": False})

print(rentals[["is_member", "insured", "paid_online"]].isna().sum())


is_member      0
insured        3
paid_online    0
dtype: int64


In [104]:
rentals.groupby("is_member")["price"].agg(["median", "count"]).round(2)

,median,count
is_member,,
False,756.25,248
True,558.00,104


### 5.

- 0 missing values for both columns before change to dates. After the change there are 2 missing in 'start_date'. Both dtypes are 'datetime'
- The min of the new column 'rental_days' is -2 which means that there are at least one rental where the end_date is before the start_date.

In [106]:
print(rentals[["start_date", "end_date"]].isna().sum())

start_date    0
end_date      0
dtype: int64


In [109]:
date_col = ["start_date", "end_date"]

for col in date_col:
    rentals[col] = pd.to_datetime(rentals[col], errors="coerce")

In [110]:
print(rentals[["start_date", "end_date"]].isna().sum())

start_date    2
end_date      0
dtype: int64


In [111]:
print(rentals[["start_date", "end_date"]].dtypes)

start_date    datetime64[us]
end_date      datetime64[us]
dtype: object


In [112]:
rentals["rental_days"] = rentals["end_date"] - rentals["start_date"]

rentals["rental_days"].describe()

count                       401
mean     2 days 22:55:21.695760
std      1 days 20:02:41.774564
min           -2 days +00:00:00
25%             1 days 00:00:00
50%             3 days 00:00:00
75%             4 days 00:00:00
max             7 days 00:00:00
Name: rental_days, dtype: object

### 6.

- There are 115 missing ratings.
- There is no perfect match between status and no_rating (Exept that that 'Cancelled' and 'No-show' are always missing a rating, which is to be expected).
- The median rating is 5

In [115]:
print(rentals["rating"].isna().sum())

115


In [116]:
no_rating = rentals["rating"].isna()

pd.crosstab(rentals["status"], no_rating)

rating,False,True
status,,
Cancelled,0,47
Completed,288,46
No-show,0,22


In [127]:
rentals["has_rating"] = ~no_rating

print(rentals["has_rating"].value_counts())

has_rating
True     288
False    115
Name: count, dtype: int64


In [130]:
rentals["rating"].describe().round(2)

count    288.00
mean       4.23
std        1.02
min        1.00
25%        4.00
50%        5.00
75%        5.00
max        5.00
Name: rating, dtype: float64

### 7.